In [ ]:
import sys
import torch

import plotly.graph_objects as go

In [2]:
sys.path.append("..")

In [17]:
from utils.grasp_utils import get_handmodel

In [18]:
from model.hand_opt import AdamGraspTransfer

## Info

In [5]:
source_gripper = "mano_right"
target_gripper = "HumanHand"
device = "cpu"

In [6]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [7]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [9]:
grasp_pose = torch.zeros(9)
grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3] = 1
grasp_pose[7] = 1 
print("Pose:", grasp_pose)


grasp_dofs_lower = source_model.dynamic_joints_q_lower.squeeze(0).clone()
grasp_dofs_mid = torch.tensor(source_model.dynamic_joints_q_mid)

# grasp_dofs = grasp_dofs_lower + grasp_dofs_mid/2
grasp_dofs = grasp_dofs_mid # for shadowhand
print("Dofs:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

Pose: tensor([0.1000, 0.2000, 0.3000, 1.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000])
Dofs: tensor([ 0.0000,  0.6981,  0.8727,  0.8727, -0.0873,  0.6981,  0.8727,  0.8727,
        -0.1745,  0.6981,  0.8727,  0.8727, -0.0873,  0.6981,  0.8727,  0.8727,
         1.2217,  0.0000,  0.8727,  0.8727])


In [10]:
sample_grasp_q.shape

torch.Size([1, 29])

In [11]:
# print("Plotting SOURCE...")
# vis_data = source_model.get_plotly_data(q=sample_grasp_q)
# fig = go.Figure(data=vis_data)
# # fig.update_layout(template='simple_white')
# # fig.update_xaxes(showgrid=False)
# # fig.update_yaxes(showgrid=False)
# fig.update_layout(
#     scene = dict(
#         xaxis = dict(visible=False),
#         yaxis = dict(visible=False),
#         zaxis =dict(visible=False)
#         )
# )
# fig.show()


In [12]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [13]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

test:   3%|▎         | 8/300 [00:00<00:04, 70.72it/s]

min energy: 0.0354
min energy index: 5


test:  55%|█████▌    | 166/300 [00:02<00:01, 78.64it/s]

min energy: 0.0336
min energy index: 20


test: 100%|██████████| 300/300 [00:04<00:00, 74.58it/s]

min energy: 0.0323
min energy index: 13


In [14]:
print(q_traj.shape)
best_q = q_traj[21, -1]
print(best_q.shape)

torch.Size([32, 301, 29])
torch.Size([29])


In [16]:
# print("Plotting TARGET and SOURCE together...")

# vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
# vis_data += target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
# fig = go.Figure(data=vis_data)
# fig.show()
# # fig.write_html("../logs_viz/gtransfer_test.html")
